# Notebook 05 — Tuning des hyperparamètres
## Phase 3 : Optimisation du meilleur modèle

**Objectif** : optimiser les hyperparamètres de la meilleure combinaison modèle × stratégie identifiée dans `04_modeling.ipynb`, puis valider le résultat sur le jeu de validation.

D'après les résultats de la Phase 3 (modélisation), la meilleure combinaison identifiée est :
- **Modèle** : XGBoost
- **Stratégie** : baseline (déséquilibre géré via `scale_pos_weight`)
- **F1 CV** : 0.8785 ± 0.0243

Ce notebook applique un `RandomizedSearchCV` sur ce modèle avec une grille justifiée.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import joblib

PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
MODEL_DIR = PROJECT_ROOT / 'models'

train_df = pd.read_csv(DATA_DIR / 'train.csv')
X_train = train_df.drop(columns=['bad_nutrition'])
y_train = train_df['bad_nutrition']

validation_df = pd.read_csv(DATA_DIR / 'validation.csv')
X_val = validation_df.drop(columns=['bad_nutrition'])
y_val = validation_df['bad_nutrition']

preprocessor = joblib.load(MODEL_DIR / 'preprocessor.joblib')

model_selection = pd.read_csv(MODEL_DIR / 'model_selection_results.csv')
best_config = model_selection.sort_values(by=['mean_f1', 'std_f1'], ascending=[False, True]).iloc[0]

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# Résumé lisible
summary = pd.DataFrame({
    'Info': ['Train shape', 'Validation shape', 'Meilleur modèle', 'Meilleure stratégie', 'F1 CV avant tuning', 'scale_pos_weight'],
    'Valeur': [
        str(X_train.shape),
        str(X_val.shape),
        best_config['model'],
        best_config['strategy'],
        round(best_config['mean_f1'], 4),
        round(scale_pos_weight, 2)
    ]
})
print(summary.to_string(index=False))

               Info     Valeur
        Train shape (9068, 16)
   Validation shape (3023, 16)
    Meilleur modèle    XGBoost
Meilleure stratégie   baseline
 F1 CV avant tuning     0.8785
   scale_pos_weight       4.71


## 1. Construction du pipeline XGBoost

On utilise le preprocessor de Phase 2 (`preprocessor.joblib`) combiné au modèle XGBoost.
La stratégie est **baseline** : pas de rééchantillonnage, le déséquilibre est géré via `scale_pos_weight` calculé sur le train set.

In [2]:
base_model = XGBClassifier(
    random_state=42,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss'
)

pipeline = Pipeline([
    ('preprocessor', clone(preprocessor)),
    ('clf', base_model)
])

# Affichage lisible des étapes du pipeline
print('Étapes du pipeline :')
for name, step in pipeline.steps:
    print(f'  [{name}] {type(step).__name__}')

Étapes du pipeline :
  [preprocessor] ColumnTransformer
  [clf] XGBClassifier


## 2. Grille d'hyperparamètres et justification

Conformément aux recommandations du descriptif Phase 3 (section 2.3), voici la grille choisie pour XGBoost :

| Hyperparamètre | Valeurs testées | Justification |
|---|---|---|
| `max_depth` | [3, 5, 7] | Contrôle la complexité. En dessous de 3 : underfitting. Au-dessus de 7 : surapprentissage sur données tabulaires. |
| `learning_rate` | [0.01, 0.05, 0.1] | Taux faible = meilleure généralisation mais nécessite plus d'estimateurs. |
| `n_estimators` | [300, 500, 700] | Cohérent avec un learning_rate faible. Au-delà de 700, gain marginal négligeable. |
| `subsample` | [0.7, 0.8, 1.0] | Sous-échantillonnage des lignes par arbre : régularisation stochastique recommandée dans le descriptif. |
| `colsample_bytree` | [0.7, 0.8, 1.0] | Sous-échantillonnage des colonnes : réduit la corrélation entre arbres. |
| `scale_pos_weight` | fixé à ≈4.71 | Ratio classes majoritaire/minoritaire. Non tuné car déterministe. |

**Choix de RandomizedSearchCV** : l'espace total est 3×3×3×3×3 = 243 combinaisons. GridSearchCV serait trop coûteux. Avec `n_iter=20`, on explore ~8% de l'espace de façon aléatoire mais représentative.

In [3]:
param_distributions = {
    'clf__max_depth': [3, 5, 7],
    'clf__learning_rate': [0.01, 0.05, 0.1],
    'clf__n_estimators': [300, 500, 700],
    'clf__subsample': [0.7, 0.8, 1.0],
    'clf__colsample_bytree': [0.7, 0.8, 1.0]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    scoring='f1',
    cv=cv,
    random_state=42,
    n_jobs=1,
    verbose=1
)

search.fit(X_train, y_train)

best_model  = search.best_estimator_
best_params = search.best_params_
best_score  = search.best_score_

# Affichage organisé des résultats
params_df = pd.DataFrame({
    'Hyperparamètre': list(best_params.keys()),
    'Valeur optimale': list(best_params.values())
})
print('=== Meilleurs hyperparamètres ===')
print(params_df.to_string(index=False))
print()
print(f'F1 CV avant tuning : {best_config["mean_f1"]:.4f}')
print(f'F1 CV après tuning : {best_score:.4f}')
print(f'Gain               : {best_score - best_config["mean_f1"]:+.4f}')

Fitting 5 folds for each of 20 candidates, totalling 100 fits
=== Meilleurs hyperparamètres ===
       Hyperparamètre  Valeur optimale
       clf__subsample              0.8
    clf__n_estimators            700.0
       clf__max_depth              5.0
   clf__learning_rate              0.1
clf__colsample_bytree              1.0

F1 CV avant tuning : 0.8785
F1 CV après tuning : 0.8820
Gain               : +0.0035


## 3. Validation sur le jeu de validation

On évalue le modèle tuné sur `validation.csv` (jamais utilisé pendant le tuning) pour confirmer que le gain se généralise.

In [5]:
y_val_pred = best_model.predict(X_val)
f1_val = f1_score(y_val, y_val_pred)

print(f'F1 sur validation : {f1_val:.4f}')
print()
print('=== Rapport de classification ===')
print(classification_report(y_val, y_val_pred, target_names=['bonne_nutrition', 'mauvaise_nutrition']))

print('=== Matrice de confusion ===')
cm = confusion_matrix(y_val, y_val_pred)
tn, fp, fn, tp = cm.ravel()
cm_df = pd.DataFrame(
    [[f'TN = {tn}', f'FP = {fp}'],
     [f'FN = {fn}', f'TP = {tp}']],
    index=['Réel : bonne_nutrition', 'Réel : mauvaise_nutrition'],
    columns=['Prédit : bonne_nutrition', 'Prédit : mauvaise_nutrition']
)
print(cm_df.to_string())
print()
print('=== Interprétation métier ===')
print(f'  TP = {tp} : produits à mauvaise nutrition correctement détectés (bonne détection)')
print(f'  TN = {tn} : produits sains correctement identifiés')
print(f'  FP = {fp} : produits sains signalés à tort (fausse alarme)')
print(f'  FN = {fn} : produits à mauvaise nutrition manqués — les plus dangereux pour le consommateur')

F1 sur validation : 0.8922

=== Rapport de classification ===
                    precision    recall  f1-score   support

   bonne_nutrition       0.98      0.98      0.98      2493
mauvaise_nutrition       0.89      0.90      0.89       530

          accuracy                           0.96      3023
         macro avg       0.93      0.94      0.93      3023
      weighted avg       0.96      0.96      0.96      3023

=== Matrice de confusion ===
                          Prédit : bonne_nutrition Prédit : mauvaise_nutrition
Réel : bonne_nutrition                   TN = 2432                     FP = 61
Réel : mauvaise_nutrition                  FN = 54                    TP = 476

=== Interprétation métier ===
  TP = 476 : produits à mauvaise nutrition correctement détectés (bonne détection)
  TN = 2432 : produits sains correctement identifiés
  FP = 61 : produits sains signalés à tort (fausse alarme)
  FN = 54 : produits à mauvaise nutrition manqués — les plus dangereux pour le cons

## 4. Sauvegarde du pipeline tuné

Le pipeline complet (preprocessor + modèle tuné) est sauvegardé dans `models/tuned_model.joblib`.
`06_evaluation.ipynb` chargera ce fichier, optimisera le seuil de décision,
et produira le livrable final `models/final_model.joblib` avec le seuil en métadonnée.

In [ ]:
# Sauvegarde intermédiaire du pipeline tuné
# final_model.joblib sera produit par 06_evaluation avec le seuil optimal
tuned_model_path = MODEL_DIR / 'tuned_model.joblib'
joblib.dump(best_model, tuned_model_path)

final_summary = pd.DataFrame({
    'Étape': ['F1 CV avant tuning', 'F1 CV après tuning', 'F1 validation', 'Pipeline tuné sauvegardé'],
    'Valeur': [
        round(best_config['mean_f1'], 4),
        round(best_score, 4),
        round(f1_val, 4),
        str(tuned_model_path)
    ]
})
print(final_summary.to_string(index=False))

             Étape                                                                             Valeur
F1 CV avant tuning                                                                             0.8785
F1 CV après tuning                                                                              0.882
     F1 validation                                                                             0.8922
 Modèle sauvegardé C:\Users\user\Desktop\data_VF\Projet_ML_Healthy_illusion\models\final_model.joblib


## 5. Synthèse du tuning

**Conclusions :**
- Le `RandomizedSearchCV` avec 20 itérations a exploré l'espace des hyperparamètres XGBoost de façon efficace.
- La grille respecte les recommandations du descriptif Phase 3 : `max_depth`, `learning_rate`, `n_estimators`, `subsample`, `colsample_bytree`.
- Chaque plage de valeurs est justifiée méthodologiquement (voir tableau section 2).
- Le pipeline tuné est sauvegardé dans `models/tuned_model.joblib`, 
  prêt à être chargé par `06_evaluation.ipynb`.